# BoTorch API Demonstration

This notebook demonstrates:
1. **Native BoTorch API** - Core components for Gaussian Process modeling and Pareto optimization
2. **Wrapper Layer** (`Botorch_utils.py`) - Higher-level abstractions for drug discovery workflows

---

## 1. Setup and Imports

In [1]:
# Core imports
import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings('ignore')

# Verify versions
import botorch
import gpytorch

print(f"PyTorch version: {torch.__version__}")
print(f"BoTorch version: {botorch.__version__}")
print(f"GPyTorch version: {gpytorch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

PyTorch version: 2.7.1
BoTorch version: 0.9.2
GPyTorch version: 1.11
Device: cpu


---
## 2. Native BoTorch API

### 2.1 Core Components

In [2]:
# Native BoTorch imports
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.utils.multi_objective.pareto import is_non_dominated
from gpytorch.mlls import ExactMarginalLogLikelihood

print("Core BoTorch components loaded:")
print("  - SingleTaskGP: Gaussian Process surrogate model")
print("  - fit_gpytorch_mll: Model fitting via MLL optimization")
print("  - is_non_dominated: Pareto dominance computation")
print("  - ExactMarginalLogLikelihood: Loss function for GP training")

Core BoTorch components loaded:
  - SingleTaskGP: Gaussian Process surrogate model
  - fit_gpytorch_mll: Model fitting via MLL optimization
  - is_non_dominated: Pareto dominance computation
  - ExactMarginalLogLikelihood: Loss function for GP training


### 2.2 SingleTaskGP - Creating and Training a GP Model

In [3]:
# Generate synthetic training data
torch.manual_seed(42)
np.random.seed(42)

n_train = 50
n_features = 3

# Training data (must be float64 for BoTorch)
X_train = torch.rand(n_train, n_features, dtype=torch.float64)

# True function: f(x) = sin(2π * x₀) + 0.5 * x₁ - x₂²
y_train = (
    torch.sin(2 * np.pi * X_train[:, 0]) + 
    0.5 * X_train[:, 1] - 
    X_train[:, 2] ** 2 +
    0.1 * torch.randn(n_train, dtype=torch.float64)  # noise
).unsqueeze(-1)  # Shape: (n_train, 1)

print(f"Training data shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  y_train: {y_train.shape}")

Training data shapes:
  X_train: torch.Size([50, 3])
  y_train: torch.Size([50, 1])


In [4]:
# Create SingleTaskGP model
gp_model = SingleTaskGP(X_train, y_train)

print("SingleTaskGP model created")
print(f"  Covariance kernel: {gp_model.covar_module}")
print(f"  Mean module: {gp_model.mean_module}")

SingleTaskGP model created
  Covariance kernel: ScaleKernel(
  (base_kernel): MaternKernel(
    (lengthscale_prior): GammaPrior()
    (raw_lengthscale_constraint): Positive()
  )
  (outputscale_prior): GammaPrior()
  (raw_outputscale_constraint): Positive()
)
  Mean module: ConstantMean()


In [5]:
# Fit the model using ExactMarginalLogLikelihood
mll = ExactMarginalLogLikelihood(gp_model.likelihood, gp_model)
fit_gpytorch_mll(mll)

print("Model fitted successfully!")
print(f"\nOptimized hyperparameters:")
print(f"  Noise variance: {gp_model.likelihood.noise.item():.4f}")
print(f"  Output scale: {gp_model.covar_module.outputscale.item():.4f}")
print(f"  Lengthscales: {gp_model.covar_module.base_kernel.lengthscale.detach().numpy().flatten()}")

Model fitted successfully!

Optimized hyperparameters:
  Noise variance: 0.0083
  Output scale: 0.5861
  Lengthscales: [0.28399736 0.93979699 0.95404944]


### 2.3 Making Predictions with GP Posterior

In [6]:
# Generate test data
n_test = 20
X_test = torch.rand(n_test, n_features, dtype=torch.float64)

# True values for test data
y_test_true = (
    torch.sin(2 * np.pi * X_test[:, 0]) + 
    0.5 * X_test[:, 1] - 
    X_test[:, 2] ** 2
)

# Make predictions
gp_model.eval()  # Set to evaluation mode
with torch.no_grad():
    posterior = gp_model.posterior(X_test)
    mean = posterior.mean.squeeze(-1)       # Predicted mean
    variance = posterior.variance.squeeze(-1)  # Predicted variance
    std = variance.sqrt()                   # Standard deviation

print(f"Predictions shape: {mean.shape}")
print(f"\nSample predictions (first 5 points):")
print(f"{'True':>10} {'Predicted':>10} {'Std':>10}")
for i in range(5):
    print(f"{y_test_true[i].item():>10.4f} {mean[i].item():>10.4f} {std[i].item():>10.4f}")

Predictions shape: torch.Size([20])

Sample predictions (first 5 points):
      True  Predicted        Std
    0.6958     0.8130     0.0966
    0.5606     0.5925     0.0870
   -0.8009    -0.8741     0.1140
   -0.5729    -0.3672     0.1936
   -0.8411    -0.8824     0.1017


### 2.4 Pareto Dominance with `is_non_dominated`

In [7]:
# Create multi-objective data
# Objective 1: Potency (higher is better)
# Objective 2: -Cost (higher is better, i.e., lower cost)

objectives = torch.tensor([
    [7.0, -0.3],   # Compound A: moderate potency, moderate cost
    [8.5, -0.5],   # Compound B: high potency, high cost
    [6.0, -0.15],  # Compound C: low potency, low cost
    [9.0, -0.8],   # Compound D: very high potency, very high cost
    [7.5, -0.25],  # Compound E: good potency, low-moderate cost
    [5.5, -0.4],   # Compound F: low potency, moderate cost (dominated)
], dtype=torch.float64)

# Find Pareto-optimal points
pareto_mask = is_non_dominated(objectives)
pareto_indices = torch.where(pareto_mask)[0]

print("Multi-objective Pareto Analysis")
print("="*50)
print(f"\n{'Compound':<10} {'Potency':<10} {'Cost':<10} {'Pareto?':<10}")
print("-"*40)
labels = ['A', 'B', 'C', 'D', 'E', 'F']
for i, (obj, is_pareto) in enumerate(zip(objectives, pareto_mask)):
    status = "✓ Yes" if is_pareto else "✗ No"
    print(f"{labels[i]:<10} {obj[0].item():<10.1f} {-obj[1].item():<10.2f} {status:<10}")

print(f"\nPareto-optimal compounds: {[labels[i] for i in pareto_indices.tolist()]}")

Multi-objective Pareto Analysis

Compound   Potency    Cost       Pareto?   
----------------------------------------
A          7.0        0.30       ✗ No      
B          8.5        0.50       ✓ Yes     
C          6.0        0.15       ✓ Yes     
D          9.0        0.80       ✓ Yes     
E          7.5        0.25       ✓ Yes     
F          5.5        0.40       ✗ No      

Pareto-optimal compounds: ['B', 'C', 'D', 'E']


---
## 3. Wrapper Layer API (`Botorch_utils`)

### 3.1 Import Wrapper Module

In [8]:
# Import the wrapper module
from Botorch_utils import (
    # Data structures
    OptimizationConfig,
    MolecularDescriptors,
    GPModelMetrics,
    ParetoResult,
    StrategyComparison,
    SelectionStrategy,
    
    # GP utilities
    train_gp_model,
    predict_with_gp,
    evaluate_gp_model,
    
    # Pareto utilities
    find_pareto_front,
    compute_pareto_result,
    
    # Selection strategies
    select_random,
    select_top_potency,
    select_lowest_cost,
    select_balanced_ratio,
    get_all_strategies,
    compare_strategies,
    
    # Pipeline functions
    prepare_optimization_data,
    run_mobo_pipeline,
    export_results,
)

print("Wrapper layer components loaded successfully!")

Wrapper layer components loaded successfully!


### 3.2 Data Structures

In [9]:
# OptimizationConfig - Configuration dataclass
config = OptimizationConfig(
    feature_cols=['mol_weight', 'logp', 'tpsa'],
    test_size=0.3,
    random_state=42,
    n_compounds_to_select=10,
    output_dir='api_demo_results'
)

print("OptimizationConfig:")
print(f"  feature_cols: {config.feature_cols}")
print(f"  test_size: {config.test_size}")
print(f"  n_compounds_to_select: {config.n_compounds_to_select}")
print(f"  output_dir: {config.output_dir}")

OptimizationConfig:
  feature_cols: ['mol_weight', 'logp', 'tpsa']
  test_size: 0.3
  n_compounds_to_select: 10
  output_dir: api_demo_results


In [10]:
# MolecularDescriptors - Container for molecular properties
descriptors = MolecularDescriptors(
    mol_weight=350.5,
    logp=2.3,
    hbd=2,
    hba=5,
    tpsa=75.2,
    sa_score=3.5
)

print("MolecularDescriptors:")
print(f"  mol_weight: {descriptors.mol_weight}")
print(f"  logp: {descriptors.logp}")
print(f"  sa_score: {descriptors.sa_score}")
print(f"\nAs dictionary: {descriptors.to_dict()}")

MolecularDescriptors:
  mol_weight: 350.5
  logp: 2.3
  sa_score: 3.5

As dictionary: {'mol_weight': 350.5, 'logp': 2.3, 'hbd': 2, 'hba': 5, 'rotatable_bonds': 0, 'heavy_atoms': 0, 'rings': 0, 'tpsa': 75.2, 'aromatic_rings': 0, 'amide_bonds': 0, 'num_heteroatoms': 0, 'sa_score': 3.5}


In [11]:
# GPModelMetrics - Model evaluation results
metrics = GPModelMetrics(
    r2_score=0.85,
    mse=0.12,
    mae=0.28,
    objective_name="Potency (pIC50)"
)

print(metrics)

Potency (pIC50) Model:
  R² Score: 0.8500
  MSE: 0.1200
  MAE: 0.2800


In [12]:
# SelectionStrategy enum
print("Available Selection Strategies:")
for strategy in SelectionStrategy:
    print(f"  - {strategy.name}: {strategy.value}")

Available Selection Strategies:
  - RANDOM: random
  - TOP_POTENCY: top_potency
  - LOWEST_COST: lowest_cost
  - BALANCED_RATIO: balanced_ratio
  - PARETO_OPTIMAL: pareto_optimal


### 3.3 GP Model Utilities

In [13]:
# Using wrapper's train_gp_model function
# Reuse the synthetic data from earlier

gp_wrapper = train_gp_model(
    X_train, 
    y_train.squeeze(-1),  # Wrapper expects 1D tensor
    objective_name="Demo Objective"
)

Training GP for Demo Objective...
✓ GP model trained for Demo Objective


In [14]:
# Using wrapper's predict_with_gp function
mean_pred, std_pred = predict_with_gp(gp_wrapper, X_test)

print(f"Predictions using wrapper:")
print(f"  Mean shape: {mean_pred.shape}")
print(f"  Std shape: {std_pred.shape}")
print(f"\nFirst 5 predictions: {mean_pred[:5].numpy().round(4)}")

Predictions using wrapper:
  Mean shape: torch.Size([20])
  Std shape: torch.Size([20])

First 5 predictions: [ 0.813   0.5925 -0.8741 -0.3672 -0.8824]


In [15]:
# Using wrapper's evaluate_gp_model function
eval_metrics = evaluate_gp_model(
    gp_wrapper,
    X_test,
    y_test_true.numpy(),
    objective_name="Demo Objective"
)

print(eval_metrics)

Demo Objective Model:
  R² Score: 0.9761
  MSE: 0.0118
  MAE: 0.0930


### 3.4 Pareto Optimization Utilities

In [16]:
# Using wrapper's find_pareto_front function
pareto_mask_w, pareto_indices_w = find_pareto_front(objectives)

print("Pareto front using wrapper:")
print(f"  Pareto mask: {pareto_mask_w.tolist()}")
print(f"  Pareto indices: {pareto_indices_w.tolist()}")

Pareto front using wrapper:
  Pareto mask: [False, True, True, True, True, False]
  Pareto indices: [1, 2, 3, 4]


### 3.5 Selection Strategies

In [17]:
# Create synthetic compound data for strategy demonstration
np.random.seed(42)
n_compounds = 100

df_demo = pd.DataFrame({
    'compound_id': [f'COMP_{i:03d}' for i in range(n_compounds)],
    'pIC50': np.random.uniform(4, 10, n_compounds),
    'cost_metric': np.random.uniform(0.1, 0.9, n_compounds),
    'mol_weight': np.random.uniform(200, 600, n_compounds),
    'logp': np.random.uniform(0, 5, n_compounds),
})

print(f"Demo compound dataset: {len(df_demo)} compounds")
df_demo.head()

Demo compound dataset: 100 compounds


,compound_id,pIC50,cost_metric,mol_weight,logp
0,COMP_000,6.247241,0.125143,456.812658,0.258409
1,COMP_001,9.704286,0.609128,233.655986,2.656773
2,COMP_002,8.391964,0.351485,264.651486,2.703176
3,COMP_003,7.591951,0.506857,559.421675,3.187150
4,COMP_004,4.936112,0.826053,442.571624,3.630457


In [18]:
# Demonstrate individual selection strategies
n_select = 5

print("Selection Strategy Demonstrations:")
print("="*60)

# Random selection
random_sel = select_random(df_demo, n_select)
print(f"\n1. RANDOM SELECTION (n={n_select})")
print(random_sel[['compound_id', 'pIC50', 'cost_metric']].to_string(index=False))

# Top potency
top_potency_sel = select_top_potency(df_demo, n_select)
print(f"\n2. TOP POTENCY (n={n_select})")
print(top_potency_sel[['compound_id', 'pIC50', 'cost_metric']].to_string(index=False))

# Lowest cost
lowest_cost_sel = select_lowest_cost(df_demo, n_select)
print(f"\n3. LOWEST COST (n={n_select})")
print(lowest_cost_sel[['compound_id', 'pIC50', 'cost_metric']].to_string(index=False))

# Balanced ratio
balanced_sel = select_balanced_ratio(df_demo, n_select)
print(f"\n4. BALANCED RATIO (n={n_select})")
print(balanced_sel[['compound_id', 'pIC50', 'cost_metric']].to_string(index=False))

Selection Strategy Demonstrations:

1. RANDOM SELECTION (n=5)
compound_id    pIC50  cost_metric
   COMP_083 4.381350     0.801871
   COMP_053 9.368964     0.491562
   COMP_070 8.633469     0.642051
   COMP_045 7.975134     0.129510
   COMP_044 5.552680     0.327872

2. TOP POTENCY (n=5)
compound_id    pIC50  cost_metric
   COMP_069 9.921322     0.572714
   COMP_011 9.819459     0.228977
   COMP_050 9.817508     0.826613
   COMP_034 9.793792     0.854328
   COMP_001 9.704286     0.609128

3. LOWEST COST (n=5)
compound_id    pIC50  cost_metric
   COMP_028 7.554487     0.105562
   COMP_071 5.192294     0.113270
   COMP_000 6.247241     0.125143
   COMP_045 7.975134     0.129510
   COMP_068 4.447304     0.132620

4. BALANCED RATIO (n=5)
compound_id    pIC50  cost_metric
   COMP_028 7.554487     0.105562
   COMP_045 7.975134     0.129510
   COMP_009 8.248435     0.161584
   COMP_052 9.636994     0.215916
   COMP_048 7.280262     0.141183


### 3.6 Strategy Comparison

In [19]:
# Create a mock Pareto compounds dataframe for demonstration
# In practice, this would come from compute_pareto_result()
pareto_demo = df_demo.nlargest(10, 'pIC50').copy()
pareto_demo['predicted_potency'] = pareto_demo['pIC50'] * 0.95  # Mock predictions

# Get all strategies
all_strategies = get_all_strategies(
    df_candidates=df_demo,
    pareto_compounds=pareto_demo,
    n_select=10,
    random_state=42
)

print("Strategies retrieved:")
for name, selected in all_strategies.items():
    print(f"  - {name}: {len(selected)} compounds")

Strategies retrieved:
  - Random: 10 compounds
  - Top Potency: 10 compounds
  - Lowest Cost: 10 compounds
  - Balanced Ratio: 10 compounds
  - Pareto-Optimal (BoTorch): 10 compounds


In [20]:
# Compare all strategies
comparison = compare_strategies(all_strategies)

print("Strategy Comparison Results:")
print("="*80)
comparison

Strategy Comparison Results:


,Strategy,Avg_Potency,Avg_Cost,Max_Potency,Min_Cost,Potency_Cost_Product
3,Balanced Ratio,8.168903,0.178736,9.819459,0.105562,43.282136
2,Lowest Cost,6.354650,0.144363,8.248435,0.105562,41.166992
1,Top Potency,9.674281,0.480368,9.921322,0.215916,19.728624
4,Pareto-Optimal (BoTorch),9.674281,0.480368,9.921322,0.215916,19.728624
0,Random,6.785475,0.445449,9.368964,0.125143,14.898419


---
## 4. Native vs Wrapper Comparison

| Task | Native BoTorch | Wrapper Layer |
|------|----------------|---------------|
| Create GP model | `SingleTaskGP(X, y)` | `train_gp_model(X, y, name)` |
| Fit model | `fit_gpytorch_mll(mll)` | (included in train_gp_model) |
| Predict | `model.posterior(X).mean` | `predict_with_gp(model, X)` |
| Find Pareto | `is_non_dominated(Y)` | `find_pareto_front(Y)` |
| Configure | Manual dict | `OptimizationConfig` dataclass |
| Evaluate | Manual metrics | `evaluate_gp_model()` → `GPModelMetrics` |

In [21]:
# Side-by-side demonstration
print("NATIVE BoTorch API:")
print("-" * 40)
print("""# Create model
model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

# Predict
model.eval()
with torch.no_grad():
    posterior = model.posterior(X_test)
    mean = posterior.mean
    std = posterior.variance.sqrt()""")

print("\n" + "="*50)
print("\nWRAPPER LAYER API:")
print("-" * 40)
print("""# Create and train model
model = train_gp_model(X_train, y_train, "Objective")

# Predict
mean, std = predict_with_gp(model, X_test)

# Evaluate
metrics = evaluate_gp_model(model, X_test, y_true, "Objective")""")

NATIVE BoTorch API:
----------------------------------------
# Create model
model = SingleTaskGP(X_train, y_train)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll)

# Predict
model.eval()
with torch.no_grad():
    posterior = model.posterior(X_test)
    mean = posterior.mean
    std = posterior.variance.sqrt()


WRAPPER LAYER API:
----------------------------------------
# Create and train model
model = train_gp_model(X_train, y_train, "Objective")

# Predict
mean, std = predict_with_gp(model, X_test)

# Evaluate
metrics = evaluate_gp_model(model, X_test, y_true, "Objective")
